In [1]:
import tensorflow as tf
from keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Flatten, Dense, Conv2D, Dropout
)
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import AUC, TopKCategoricalAccuracy
import h5py
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import keras
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

best_params = None


/Users/logan/brain_strain_cnn/rugby_brain_strain_CNN/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [4]:
import h5py
import numpy as np
from sklearn.model_selection import train_test_split
import glob
import os

data_folder = '/scratch/projects/MADS2025-CiJR/rugby_brain_strain_CNN/data/cghs'

# Make sure the path exists
if not os.path.exists(data_folder):
    print("Folder does not exist:", data_folder)
else:
    # Recursive search for all .h5 files
    file_list = glob.glob(os.path.join(data_folder, '**', '*.h5'), recursive=True)
    print(f"Found {len(file_list)} files:")
    for f in file_list:
        print(f)


X = []
y = []

for file_path in file_list:
    with h5py.File(file_path, 'r') as hf:
        for key in hf.keys():
            group = hf[key]
            pred = group.attrs['QA']

            dset_names = list(group.keys())
            perm_names = [n for n in dset_names if n.startswith('perm_') and not n.startswith('lin_perm_')]

            for perm_name in perm_names:
                lin_name = perm_name.replace('perm_', 'lin_perm_', 1)
                if lin_name not in group:
                    continue

                rot = group[perm_name][:]
                lin = group[lin_name][:]

                if rot.shape != lin.shape or rot.ndim != 3 or rot.shape[0] != 1:
                    continue

                # Convert (1, 3, L) -> (3, L, 1) and stack as channels
                rot = rot.transpose(1, 2, 0)
                lin = lin.transpose(1, 2, 0)
                sample = np.concatenate([rot, lin], axis=2)  # (3, L, 2)

                X.append(sample)
                y.append(pred)

# Convert to numpy arrays
if len(X) > 0:
    X = np.stack(X, axis=0)
    y = np.array(y)
else:
    X = np.array([])
    y = np.array([])

# 80:20 train-test split
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
        stratify=y
    )

    input_shape = X_train.shape[1:]

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

else:
    X_train, X_test = [], []
    y_train, y_test = [], []
    input_shape = (3, 1000, 2)


Found 2 files:
/scratch/projects/MADS2025-CiJR/rugby_brain_strain_CNN/data/cghs/cghs_game.h5
/scratch/projects/MADS2025-CiJR/rugby_brain_strain_CNN/data/cghs/cghs_training.h5


ValueError: axes don't match array

In [ ]:
# 10-fold CV to tune filters/kernel/stride using PR-AUC (best for imbalance)

def build_model(params, input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel1"],
            strides=params["stride1"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel2"],
            strides=params["stride2"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel3"],
            strides=params["stride3"],
            activation='relu',
            padding='valid'
        ),
        Dropout(params.get("dropout", 0.2)),
        Flatten(),
        Dense(units=10, activation='relu'),
        Dense(units=1, activation='sigmoid')
    ])

    optimizer = Adam(learning_rate=params.get("lr", 1e-4))
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
        ]
    )
    return model

param_grid = [
    {
        "filters": 16,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
    },
    {
        "filters": 32,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
    },
    {
        "filters": 32,
        "kernel1": (3, 7),
        "stride1": (1, 2),
        "kernel2": (1, 7),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
    },
]

if len(X_train) > 0:
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_results = []

    for params in param_grid:
        fold_scores = []
        for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            # Handle class imbalance with weights
            y_tr_int = y_tr.astype(int)
            neg = (y_tr_int == 0).sum()
            pos = (y_tr_int == 1).sum()
            class_weight = None
            if pos > 0:
                class_weight = {0: 1.0, 1: neg / pos}

            model = build_model(params, input_shape=X_train.shape[1:])

            early_stopping = EarlyStopping(
                monitor='val_pr_auc',
                patience=8,
                mode='max',
                restore_best_weights=True
            )

            model.fit(
                X_tr,
                y_tr,
                epochs=60,
                batch_size=64,
                validation_data=(X_val, y_val),
                callbacks=[early_stopping],
                class_weight=class_weight,
                verbose=0
            )

            y_pred = model.predict(X_val, verbose=0).ravel()
            pr_auc = average_precision_score(y_val, y_pred)
            fold_scores.append(pr_auc)

        mean_pr_auc = float(np.mean(fold_scores))
        cv_results.append((mean_pr_auc, params))
        print(f"Params {params} -> mean PR-AUC: {mean_pr_auc:.4f}")

    best_pr_auc, best_params = max(cv_results, key=lambda x: x[0])
    print(f"Best params: {best_params}")
    print(f"Best mean PR-AUC: {best_pr_auc:.4f}")
else:
    best_params = None



The number of CNN filters and their sizes and stride sizes were iteratively and empirically updated until a 10-fold cross-validation performance was maximized in terms of R2 between the predicted and directly simulated responses.


In [ ]:
input_shape = X_train.shape[1:] if len(X_train) else input_shape

if "build_model" not in globals():
    def build_model(params, input_shape):
        model = Sequential([
            Input(shape=input_shape),
            Conv2D(
                filters=params["filters"],
                kernel_size=params["kernel1"],
                strides=params["stride1"],
                activation='relu',
                padding='valid'
            ),
            Conv2D(
                filters=params["filters"],
                kernel_size=params["kernel2"],
                strides=params["stride2"],
                activation='relu',
                padding='valid'
            ),
            Conv2D(
                filters=params["filters"],
                kernel_size=params["kernel3"],
                strides=params["stride3"],
                activation='relu',
                padding='valid'
            ),
            Dropout(params.get("dropout", 0.2)),
            Flatten(),
            Dense(units=10, activation='relu'),
            Dense(units=1, activation='sigmoid')
        ])

        optimizer = Adam(learning_rate=params.get("lr", 1e-4))
        model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=[
                tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
                tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
            ]
        )
        return model

default_params = {
    "filters": 32,
    "kernel1": (3, 10),
    "stride1": (1, 2),
    "kernel2": (1, 10),
    "stride2": (1, 2),
    "kernel3": (1, 5),
    "stride3": (1, 1),
}

if "best_params" in globals() and best_params:
    model = build_model(best_params, input_shape)
else:
    model = build_model(default_params, input_shape)

model.summary()



In [ ]:
input_shape = X_train.shape[1:] if len(X_train) else input_shape

model = Sequential([
    Input(shape=input_shape),
    Conv2D(
        filters=32,
        kernel_size=(3,10),
        strides=(1,2),
        activation='relu',
        padding='valid'
    ),
    Conv2D(
        filters=32,
        kernel_size=(1,10),
        strides=(1,2),
        activation='relu',
        padding='valid'
    ),
    Conv2D(
        filters=32,
        kernel_size=(1,5),
        strides=(1,1),
        activation='relu',
        padding='valid'
    ),
    Dropout(0.2),
    Flatten(),
    Dense(
        units=10,
        activation='relu'
    ),
    Dense(
        units=1,
        activation='sigmoid'
    )
])

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 1, 496, 32)     │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 1, 244, 32)     │        10,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 1, 240, 32)     │         5,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 240, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 7680)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │        76,810 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,237 (364.21 KB)

 Trainable params: 93,237 (364.21 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Early stopping callback (validation-based)
early_stopping = EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=20,
    restore_best_weights=True
)

# Model training
history = model.fit(
    X_train,
    y_train,
    epochs=250,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    shuffle=True
)



Epoch 1/250


2026-01-07 14:07:06.238856: I external/local_xla/xla/service/service.cc:163] XLA service 0x151058017ee0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-01-07 14:07:06.238876: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA L40S, Compute Capability 8.9
2026-01-07 14:07:06.279984: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-07 14:07:06.512766: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91701


   90/10410 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy: 0.4243 - loss: 10.1547

I0000 00:00:1767748028.733926  200044 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10410/10410 ━━━━━━━━━━━━━━━━━━━━ 35s 3ms/step - accuracy: 0.7759 - loss: 0.9981 - val_accuracy: 0.8089 - val_loss: 0.4749
Epoch 2/250
10410/10410 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8087 - loss: 0.4694 - val_accuracy: 0.8091 - val_loss: 0.4247
Epoch 3/250
10410/10410 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8090 - loss: 0.4336 - val_accuracy: 0.8091 - val_loss: 0.4140
Epoch 4/250
10410/10410 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8090 - loss: 0.4200 - val_accuracy: 0.8091 - val_loss: 0.4086
Epoch 5/250
10410/10410 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8090 - loss: 0.4130 - val_accuracy: 0.8091 - val_loss: 0.4057
Epoch 6/250
10410/10410 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8100 - loss: 0.4083 - val_accuracy: 0.8139 - val_loss: 0.4036
Epoch 7/250
 1657/10410 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.8131 - loss: 0.4082

KeyboardInterrupt: 

In [ ]:
model.save('cnn_location.keras')

In [4]:
loaded_model = keras.saving.load_model("brain_strain_cnn.keras")
